# Shadow Revenue Detection System

# Gold Layer

## Objective

The Gold layer creates business-ready datasets for revenue analysis and dashboard reporting.

This notebook compares two pipelines:

- **Bad Pipeline** – Uses uncleaned data.
- **Good Pipeline** – Uses cleaned and validated data.

The outputs from this notebook will be used to identify revenue leakage, detect anomalies, and calculate business KPIs.

In [0]:
CATALOG = "shadow_revenue_catalog"

SILVER = "silver"

GOLD = "gold"

## Load Silver Data

Read both the Bad and Good Silver datasets. These datasets will be used to calculate revenue and compare the impact of data quality improvements.

In [0]:
# Bad Pipeline
orders_bad = spark.table(f"{CATALOG}.{SILVER}.orders_bad")
products_bad = spark.table(f"{CATALOG}.{SILVER}.products_bad")
payments_bad = spark.table(f"{CATALOG}.{SILVER}.payments_bad")
customers_bad = spark.table(f"{CATALOG}.{SILVER}.customers_bad")

# Good Pipeline
orders_good = spark.table(f"{CATALOG}.{SILVER}.orders_good")
products_good = spark.table(f"{CATALOG}.{SILVER}.products_good")
payments_good = spark.table(f"{CATALOG}.{SILVER}.payments_good")
customers_good = spark.table(f"{CATALOG}.{SILVER}.customers_good")

## Create Product Lookup

Extract the latest product price for revenue calculation.

The product catalog follows Slowly Changing Dimension (SCD Type 2). Therefore, only the current active product records are considered.

In [0]:
product_bad = (

    products_bad

    .select(

        "product_id",

        col("price").alias("product_price")

    )

)

In [0]:
product_good = (

    products_good

    .filter(col("is_current") == 1)

    .select(

        "product_id",

        col("price").alias("product_price")

    )

)

## Create Revenue Fact Table - Bad Pipeline

Join orders, products, payments, and customers to calculate revenue using the uncleaned pipeline.

In [0]:
fact_revenue_bad = (
    orders_bad.alias("o")
    .join(product_bad.alias("p"), "product_id", "left")
    .join(payments_bad.alias("pay"), "order_id", "left")
    .join(customers_bad.alias("c"), "customer_id", "left")
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("o.product_id"),
        col("c.name"),
        col("c.city"),
        col("o.order_date"),
        col("o.order_status"),
        col("o.channel"),
        col("o.quantity"),
        col("o.price").alias("order_price"),
        col("p.product_price"),
        col("pay.payment_amount"),
        col("pay.payment_date")
    )
    .withColumn("calculated_revenue", col("quantity") * col("product_price"))
)


## Create Revenue Fact Table - Good Pipeline

Generate the optimized revenue fact table using the cleaned Silver datasets.

In [0]:
fact_revenue_good = (
    orders_good.alias("o")
    .join(product_good.alias("p"), "product_id", "left")
    .join(payments_good.alias("pay"), "order_id", "left")
    .join(customers_good.alias("c"), "customer_id", "left")
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("o.product_id"),
        col("c.name"),
        col("c.city"),
        col("o.order_date"),
        col("o.order_status"),
        col("o.channel"),
        col("o.quantity"),
        col("o.price").alias("order_price"),
        col("p.product_price"),
        col("pay.payment_amount"),
        col("pay.payment_date")
    )
    .withColumn("calculated_revenue", col("quantity") * col("product_price"))
)


## Save Revenue Fact Tables

Store both revenue fact tables in the Gold layer for downstream analysis.

In [0]:
fact_revenue_bad.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(f"{CATALOG}.{GOLD}.fact_revenue_bad")


fact_revenue_good.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(f"{CATALOG}.{GOLD}.fact_revenue_good")

In [0]:
print("Bad Records :", fact_revenue_bad.count())

print("Good Records :", fact_revenue_good.count())

Bad Records : 20400
Good Records : 20000


## Detect Revenue Anomalies

This section identifies common revenue anomalies such as missing payments, price mismatches, and revenue leakage. These anomaly flags will be used to calculate KPIs and build the final dashboard.

In [0]:
# Add Anomaly Flags - Bad Pipeline
fact_revenue_bad = (
    fact_revenue_bad
    # Missing Payment
    .withColumn("missing_payment", when(col("payment_amount").isNull(), 1).otherwise(0))
    # Price Mismatch
    .withColumn("price_mismatch", when(col("order_price") != col("product_price"), 1).otherwise(0))
    # Revenue Difference
    .withColumn("revenue_difference", col("calculated_revenue") - coalesce(col("payment_amount"), lit(0)))
    # Revenue Leakage
    .withColumn("revenue_leakage", when(col("calculated_revenue") > coalesce(col("payment_amount"), lit(0)), 1).otherwise(0))
)


In [0]:
# Add Anomaly Flags - Good Pipeline
fact_revenue_good = (
    fact_revenue_good
    .withColumn("missing_payment", when(col("payment_amount").isNull(), 1).otherwise(0))
    .withColumn("price_mismatch", when(col("order_price") != col("product_price"), 1).otherwise(0))
    .withColumn("revenue_difference", col("calculated_revenue") - coalesce(col("payment_amount"), lit(0)))
    .withColumn("revenue_leakage", when(col("calculated_revenue") > coalesce(col("payment_amount"), lit(0)), 1).otherwise(0))
)


## Detect Orphan Payments

Identify payments that do not have a matching order. These transactions represent revenue that cannot be linked to any valid sale.

In [0]:
# Orphan Payments
orphan_payment_bad = (
    payments_bad
    .join(orders_bad, "order_id", "left_anti")
)

orphan_payment_good = (
    payments_good
    .join(orders_good, "order_id", "left_anti")
)


## Count Duplicate Orders

The Good pipeline removes duplicate orders during the Silver layer. The difference in row counts between the Bad and Good pipelines represents the number of duplicate orders removed.

In [0]:
duplicate_orders_bad = (
    orders_bad.count()
    - orders_good.count()
)

print("Duplicate Orders Removed :", duplicate_orders_bad)


Duplicate Orders Removed : 400


## Save Updated Revenue Fact Tables

Overwrite the Gold fact tables with the enriched versions containing anomaly indicators.

In [0]:
fact_revenue_bad.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable(f"{CATALOG}.{GOLD}.fact_revenue_bad")

fact_revenue_good.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable(f"{CATALOG}.{GOLD}.fact_revenue_good")

In [0]:
display(fact_revenue_bad.limit(10))

display(fact_revenue_good.limit(10))

order_id,customer_id,product_id,name,city,order_date,order_status,channel,quantity,order_price,product_price,payment_amount,payment_date,calculated_revenue,missing_payment,price_mismatch,revenue_difference,revenue_leakage
1,1950,29,Cust_1950,Pune,2024-02-29,Cancelled,Web,4,1701.72,1136.85,532.61,2024-02-26,4547.4,0,1,4014.7899999999995,1
2,1491,98,Cust_1491,Jaipur,2024-01-28,Completed,Store,5,1643.01,1658.33,1583.67,2024-01-23,8291.65,0,1,6707.98,1
3,1518,2,Cust_1518,Jaipur,2024-01-10,Completed,Store,4,1948.47,1837.12,1271.92,2024-01-29,7348.48,0,1,6076.5599999999995,1
4,440,33,Cust_440,Bangalore,2024-02-01,Completed,Web,3,1911.69,1071.5,1296.86,2024-02-28,3214.5,0,1,1917.64,1
5,1011,1,Cust_1011,Pune,2024-02-11,Completed,App,4,795.88,1781.59,614.5,2024-01-29,7126.36,0,1,6511.86,1
6,741,31,Cust_741,Bangalore,2024-01-31,Completed,Store,3,625.85,1678.43,1579.44,2024-01-13,5035.29,0,1,3455.85,1
7,201,35,Cust_201,Pune,2024-01-14,Completed,Web,3,174.15,1507.39,1755.48,2024-02-14,4522.17,0,1,2766.69,1
8,490,100,Cust_490,Pune,2024-02-29,Cancelled,Web,5,823.93,369.61,null,null,1848.0500000000002,1,1,1848.0500000000002,1
9,269,39,Cust_269,Bangalore,2024-02-10,Cancelled,Web,1,1880.09,278.64,1056.5,2024-02-20,278.64,0,1,-777.86,0
10,1203,50,Cust_1203,Jaipur,2024-01-12,Completed,App,3,522.32,570.28,816.05,2024-01-03,1710.84,0,1,894.79,1


order_id,customer_id,product_id,name,city,order_date,order_status,channel,quantity,order_price,product_price,payment_amount,payment_date,calculated_revenue,missing_payment,price_mismatch,revenue_difference,revenue_leakage
1,1950,29,Cust_1950,Pune,2024-02-29,Cancelled,Web,4,1701.7200,1136.8500,532.6100,2024-02-26,4547.4000,0,1,4014.7900,1
2,1491,98,Cust_1491,Jaipur,2024-01-28,Completed,Store,5,1643.0100,1658.3300,1583.6700,2024-01-23,8291.6500,0,1,6707.9800,1
3,1518,2,Cust_1518,Jaipur,2024-01-10,Completed,Store,4,1948.4700,1837.1200,1271.9200,2024-01-29,7348.4800,0,1,6076.5600,1
4,440,33,Cust_440,Bangalore,2024-02-01,Completed,Web,3,1911.6900,1071.5000,1296.8600,2024-02-28,3214.5000,0,1,1917.6400,1
5,1011,1,Cust_1011,Pune,2024-02-11,Completed,App,4,795.8800,1781.5900,614.5000,2024-01-29,7126.3600,0,1,6511.8600,1
6,741,31,Cust_741,Bangalore,2024-01-31,Completed,Store,3,625.8500,1678.4300,1579.4400,2024-01-13,5035.2900,0,1,3455.8500,1
7,201,35,Cust_201,Pune,2024-01-14,Completed,Web,3,174.1500,1507.3900,1755.4800,2024-02-14,4522.1700,0,1,2766.6900,1
8,490,100,Cust_490,Pune,2024-02-29,Cancelled,Web,5,823.9300,369.6100,null,null,1848.0500,1,1,1848.0500,1
9,269,39,Cust_269,Bangalore,2024-02-10,Cancelled,Web,1,1880.0900,278.6400,1056.5000,2024-02-20,278.6400,0,1,-777.8600,0
10,1203,50,Cust_1203,Jaipur,2024-01-12,Completed,App,3,522.3200,570.2800,816.0500,2024-01-03,1710.8400,0,1,894.7900,1


# KPI Calculation

Calculate business KPIs from the Gold fact tables and compare the Bad and Good pipelines.

These KPI tables will be used directly in the dashboard.

## Calculate KPIs - Bad Pipeline

In [0]:
# KPI - Bad Pipeline
kpi_revenue_bad = (
    fact_revenue_bad
    .agg(
        sum("calculated_revenue").alias("total_revenue"),
        sum("payment_amount").alias("total_payment"),
        sum("revenue_difference").alias("total_revenue_difference"),
        sum("missing_payment").alias("missing_payments"),
        sum("price_mismatch").alias("price_mismatches"),
        sum("revenue_leakage").alias("revenue_leakages")
    )
    .withColumn("orphan_payments", lit(orphan_payment_bad.count()))
    .withColumn("duplicate_orders", lit(duplicate_orders_bad))
    .withColumn(
        "accuracy_ratio",
        when(col("total_revenue") == 0, lit(0))
        .otherwise(round((col("total_payment") / col("total_revenue")) * 100, 2))
    )
    .withColumn("pipeline", lit("Bad"))
)


## Calculate KPIs - Good Pipeline

In [0]:
# KPI - Good Pipeline
kpi_revenue_good = (
    fact_revenue_good
    .agg(
        sum("calculated_revenue").alias("total_revenue"),
        sum("payment_amount").alias("total_payment"),
        sum("revenue_difference").alias("total_revenue_difference"),
        sum("missing_payment").alias("missing_payments"),
        sum("price_mismatch").alias("price_mismatches"),
        sum("revenue_leakage").alias("revenue_leakages")
    )
    .withColumn("orphan_payments", lit(orphan_payment_good.count()))
    .withColumn("duplicate_orders", lit(0))
    .withColumn(
        "accuracy_ratio",
        when(col("total_revenue") == 0, lit(0))
        .otherwise(round((col("total_payment") / col("total_revenue")) * 100, 2))
    )
    .withColumn("pipeline", lit("Good"))
)


## Save KPI Tables

In [0]:
kpi_revenue_bad.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.format("delta") \
.saveAsTable(f"{CATALOG}.{GOLD}.kpi_revenue_bad")


kpi_revenue_good.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.format("delta") \
.saveAsTable(f"{CATALOG}.{GOLD}.kpi_revenue_good")

## Create KPI Comparison

In [0]:
kpi_comparison = (
    kpi_revenue_bad
    .select(
        "pipeline",
        "total_revenue",
        "total_payment",
        "total_revenue_difference",
        "accuracy_ratio",
        "missing_payments",
        "orphan_payments",
        "price_mismatches",
        "duplicate_orders",
        "revenue_leakages"
    )
    .unionByName(
        kpi_revenue_good.select(
            "pipeline",
            "total_revenue",
            "total_payment",
            "total_revenue_difference",
            "accuracy_ratio",
            "missing_payments",
            "orphan_payments",
            "price_mismatches",
            "duplicate_orders",
            "revenue_leakages"
        )
    )
)


In [0]:
# Save comparison
kpi_comparison.write \
.mode("overwrite") \
.option("overwriteSchema","true") \
.format("delta") \
.saveAsTable(f"{CATALOG}.{GOLD}.kpi_comparison")

## Validate Gold Layer

In [0]:
display(kpi_revenue_bad)

display(kpi_revenue_good)

display(kpi_comparison)

total_revenue,total_payment,total_revenue_difference,missing_payments,price_mismatches,revenue_leakages,orphan_payments,duplicate_orders,accuracy_ratio,pipeline
6.573054318000072E7,1.9450610200000014E7,4.627993297999992E7,2037,20400,16595,600,400,29.59,Bad


total_revenue,total_payment,total_revenue_difference,missing_payments,price_mismatches,revenue_leakages,orphan_payments,duplicate_orders,accuracy_ratio,pipeline
64379883.9800,19067003.9600,45312880.0200,2000,20000,16266,600,0,29.62,Good


pipeline,total_revenue,total_payment,total_revenue_difference,accuracy_ratio,missing_payments,orphan_payments,price_mismatches,duplicate_orders,revenue_leakages
Bad,6.573054318000072E7,1.9450610200000014E7,4.627993297999992E7,29.59,2037,600,20400,400,16595
Good,6.437988398E7,1.906700396E7,4.531288002E7,29.62,2000,600,20000,0,16266


In [0]:
%sql
SHOW TABLES IN shadow_revenue_catalog.gold

database,tableName,isTemporary
gold,fact_revenue_bad,false
gold,fact_revenue_good,false
gold,kpi_comparison,false
gold,kpi_revenue_bad,false
gold,kpi_revenue_good,false
